# PixelChat GPU backend (Colab)
1. Runtime → Change runtime type → **T4 GPU**
2. Left sidebar → 🔑 **Secrets**: add `API_TOKEN` (any password you invent) and `NGROK_TOKEN` (free, ngrok.com), and turn **Notebook access** ON for both
3. Run the cells in order. Copy the `PUBLIC URL` printed by the last cell into your Streamlit `BACKEND_URL`.

In [ ]:
!pip -q install fastapi uvicorn python-multipart pyngrok nest_asyncio diffusers transformers accelerate opencv-python-headless safetensors peft

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile /content/server.py
"""GPU backend: SDXL (RealVisXL) + ControlNet-Canny + IP-Adapter + GroundingDINO/SAM masks.
Runs on any 15-16GB CUDA GPU (Colab T4, Kaggle T4/P100) using fp16 + model CPU offload + VAE tiling (~9-12 GB peak)."""
import gc, ctypes, io, json, os, threading
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less VRAM fragmentation on T4
import cv2, numpy as np, torch
from PIL import Image, ImageFilter
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from diffusers import (AutoencoderKL, ControlNetModel, DPMSolverMultistepScheduler,
                       StableDiffusionXLControlNetInpaintPipeline)
from transformers import (AutoModelForZeroShotObjectDetection, AutoProcessor, CLIPVisionModelWithProjection,
                          SamModel, SamProcessor)

TOKEN = os.getenv("API_TOKEN", "").strip()
BASE = os.getenv("BASE_MODEL", "SG161222/RealVisXL_V4.0")   # swap for any SDXL checkpoint
CN = os.getenv("CONTROLNET", "diffusers/controlnet-canny-sdxl-1.0-small")
MAX_SIDE = int(os.getenv("MAX_SIDE", "896"))        # lower (768) if you ever hit CUDA out-of-memory
OFFLOAD = os.getenv("OFFLOAD", "0") == "1"           # 1 = CPU offload (less VRAM, but ~12 GB of system RAM)
dtype = torch.float16
lock = threading.Lock()
app = FastAPI(title="Image Edit Backend")

print("Loading models...")
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype)
cnet = ControlNetModel.from_pretrained(CN, torch_dtype=dtype)
# lighter ViT-H image encoder (~1.3 GB fp16) instead of the 3.7 GB bigG one; pairs with the vit-h adapter below
enc = CLIPVisionModelWithProjection.from_pretrained("h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=dtype)
pipe = StableDiffusionXLControlNetInpaintPipeline.from_pretrained(
    BASE, controlnet=cnet, vae=vae, image_encoder=enc, torch_dtype=dtype, variant="fp16",
    use_safetensors=True, low_cpu_mem_usage=True)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl_vit-h.bin")
if OFFLOAD:
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cuda")          # weights live on the GPU, so system RAM is freed (~3 GB instead of ~12 GB)
pipe.vae.enable_tiling()
# Grounding DINO finds the object (text -> box), SAM cuts it out (box -> pixel mask)
DINO_ID = os.getenv("DINO_MODEL", "IDEA-Research/grounding-dino-tiny")   # use ...-base for harder scenes
dino_proc = AutoProcessor.from_pretrained(DINO_ID)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to("cuda").eval()
sam_proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam = SamModel.from_pretrained("facebook/sam-vit-base").to("cuda").eval()


def free_mem():
    """Release Python, CUDA and glibc-held memory after every request."""
    gc.collect(); torch.cuda.empty_cache()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass


free_mem()
print("Ready.")

QUALITY = ", photorealistic, natural lighting, consistent shadows and perspective, sharp detail, 8k photo"
NEG = "cartoon, painting, illustration, blurry, deformed, low quality, watermark, text, artifacts, seams"


def fit(img: Image.Image):
    w, h = img.size
    s = min(1.0, MAX_SIDE / max(w, h))
    return img.resize((max(64, int(w * s) // 8 * 8), max(64, int(h * s) // 8 * 8)), Image.LANCZOS)


def detect_box(img, text):
    """Grounding DINO: best box for the text phrase, or None."""
    text = text.lower().strip().rstrip(".") + "."
    inp = dino_proc(images=img, text=text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = dino(**inp)
    for thr in (0.35, 0.25, 0.15):
        kw = dict(text_threshold=0.25, target_sizes=[img.size[::-1]])
        try:
            res = dino_proc.post_process_grounded_object_detection(out, inp["input_ids"], threshold=thr, **kw)[0]
        except TypeError:                      # older transformers name
            res = dino_proc.post_process_grounded_object_detection(out, inp["input_ids"], box_threshold=thr, **kw)[0]
        if len(res["boxes"]):
            return res["boxes"][int(res["scores"].argmax())].cpu().tolist()
    return None


def sam_mask(img, box):
    inp = sam_proc(img, input_boxes=[[box]], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = sam(**inp, multimask_output=False)
    m = sam_proc.image_processor.post_process_masks(
        out.pred_masks.cpu(), inp["original_sizes"].cpu(), inp["reshaped_input_sizes"].cpu())[0]
    return m[0, 0].numpy().astype(np.uint8) * 255


def grow(m, frac):
    k = max(3, int(frac * max(m.shape))) | 1
    return cv2.dilate(m, np.ones((k, k), np.uint8))


def build_mask(img, op, target, region):
    """add: small box placed relative to the anchor object (e.g. eye band of a face).
    remove/replace/recolor: SAM cut-out of the object itself."""
    box = detect_box(img, target)
    if box is None:
        raise HTTPException(422, f"Couldn't find '{target}' in the image. Try naming it differently.")
    w, h = img.size
    if op == "add":
        x0, y0, x1, y1 = box
        if region:
            bw, bh = x1 - x0, y1 - y0
            rx0, ry0, rx1, ry1 = region
            x0, y0, x1, y1 = x0 + rx0 * bw, y0 + ry0 * bh, x0 + rx1 * bw, y0 + ry1 * bh
        m = np.zeros((h, w), np.uint8)
        cv2.rectangle(m, (max(0, int(x0)), max(0, int(y0))), (min(w - 1, int(x1)), min(h - 1, int(y1))), 255, -1)
        g = 0.01
    else:
        m = sam_mask(img, box)
        g = {"remove": 0.04, "recolor": 0.01}.get(op, 0.03)     # remove grows more to catch shadows
    if m.sum() == 0:
        raise HTTPException(422, f"Couldn't outline '{target}'. Try a different description.")
    return Image.fromarray(grow(m, g))


def canny(img, mask, keep_inside):
    e = cv2.Canny(np.array(img.convert("L")), 80, 180)
    if not keep_inside:                       # free the masked area so new content isn't constrained
        e[np.array(mask) > 127] = 0
    return Image.fromarray(np.stack([e] * 3, -1))


@app.get("/health")
def health():
    info = {"ok": True, "gpu": torch.cuda.get_device_name(0),
            "vram_used_gb": round(torch.cuda.memory_allocated() / 2**30, 1)}
    try:
        import psutil
        info["ram_used_gb"] = round(psutil.virtual_memory().used / 2**30, 1)
        info["ram_total_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
    except Exception: pass
    return info


@app.get("/check")
def check(x_api_token: str = Header("")):
    if TOKEN and x_api_token.strip() != TOKEN:
        raise HTTPException(401, "bad token")
    return {"ok": True, "token_required": bool(TOKEN)}


@app.post("/edit")
async def edit(image: UploadFile = File(...), reference: UploadFile | None = File(None),
               op: str = Form(...), target: str = Form(""), prompt: str = Form(""),
               seed: int = Form(-1), region: str = Form(""), x_api_token: str = Header("")):
    if TOKEN and x_api_token.strip() != TOKEN:
        raise HTTPException(401, "bad token")
    orig = Image.open(io.BytesIO(await image.read())).convert("RGB")
    ref = Image.open(io.BytesIO(await reference.read())).convert("RGB") if reference else None
    img = fit(orig)
    gen = torch.Generator("cuda").manual_seed(seed if seed >= 0 else int(torch.randint(0, 2**31, (1,))))

    try:
      with lock, torch.inference_mode():
        try:
            reg = json.loads(region) if region else None
            reg = [float(v) for v in reg] if reg and len(reg) == 4 else None
        except Exception:
            reg = None
        mask = build_mask(img, op, target or prompt, reg)
        neg, ip_scale = NEG, 0.0
        if op == "remove":
            p, strength, cn_scale, keep, steps = "empty clean background, seamless natural texture, nothing there" + QUALITY, 0.99, 0.5, False, 35
            neg += f", {target}"
        elif op == "recolor":
            p, strength, cn_scale, keep, steps = f"{prompt}, same shape and texture, realistic material" + QUALITY, 0.85, 0.85, True, 35
        else:  # add / replace
            p, strength, cn_scale, keep, steps = f"{prompt}" + QUALITY, 0.95, 0.4, False, 40
            if op == "replace":
                neg += f", {target}"
        if ref is not None:
            ip_scale = 0.6
        pipe.set_ip_adapter_scale(ip_scale)
        out = pipe(prompt=p, negative_prompt=neg, image=img, mask_image=mask,
                   control_image=canny(img, mask, keep), strength=strength,
                   controlnet_conditioning_scale=cn_scale, num_inference_steps=steps,
                   guidance_scale=6.5, ip_adapter_image=ref or Image.new("RGB", (224, 224), "black"),
                   padding_mask_crop=48, generator=gen).images[0]
    finally:
        free_mem()

    # keep untouched pixels identical: feathered composite back onto the original
    out = out.resize(orig.size, Image.LANCZOS)
    soft = mask.resize(orig.size).filter(ImageFilter.GaussianBlur(6))
    final = Image.composite(out, orig, soft)
    buf = io.BytesIO(); final.save(buf, "PNG")
    return Response(buf.getvalue(), media_type="image/png")


In [ ]:
import os, sys, time, threading, uvicorn, nest_asyncio
if "server" in sys.modules:
    raise RuntimeError("Server already running. Runtime -> Restart session, then run all cells again.")
from pyngrok import ngrok
from google.colab import userdata

os.environ["API_TOKEN"] = userdata.get("API_TOKEN").strip()
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
nest_asyncio.apply()
sys.path.insert(0, "/content")

import server   # loads the models (first run downloads ~10 GB, takes a few minutes)

ngrok.kill()
print("PUBLIC URL ->", ngrok.connect(8000).public_url)
cfg = uvicorn.Config(server.app, host="0.0.0.0", port=8000, log_level="info")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()

while True:      # keep the cell (and backend) alive; keep this browser tab open
    time.sleep(60)